# Kaggle Qwen Worker

This notebook is a best-effort outbound worker. Set `GATEWAY_WS_URL` to your public `wss://.../workers/connect` URL and set `WORKER_TOKEN` if the gateway requires it.

In [ ]:
!pip install -q websockets transformers accelerate bitsandbytes httpx

In [ ]:
import asyncio
import json
import os
import shlex
import subprocess
import sys
import time
import traceback
import uuid
from threading import Thread
from urllib.parse import urlsplit

import httpx

EMBEDDED_RUNTIME_CONFIG = {}
runtime_config = dict(EMBEDDED_RUNTIME_CONFIG)
for config_path in ("worker_config.json", "/kaggle/working/worker_config.json"):
    if os.path.exists(config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            runtime_config.update(json.load(f))
        print({"loaded_worker_config": config_path}, flush=True)
        break

def config_value(name, default=""):
    return os.environ.get(name) or runtime_config.get(name) or default

GATEWAY_WS_URL = config_value("GATEWAY_WS_URL", "wss://your-domain.example/workers/connect")
WORKER_TOKEN = config_value("WORKER_TOKEN", "")
OWNER = os.environ.get("KAGGLE_USERNAME") or runtime_config.get("OWNER") or "unknown"
NODE_ID = os.environ.get("NODE_ID", f"kaggle-{OWNER}-{uuid.uuid4().hex[:8]}")
SERVED_MODEL = config_value("SERVED_MODEL", "qwen2.5-9b-quantized")
MODEL_ID = config_value("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")
WORKER_BACKEND = str(config_value("WORKER_BACKEND", "vllm")).lower()
LOAD_IN_4BIT = config_value("LOAD_IN_4BIT", "true").lower() in {"1", "true", "yes", "y"}
HF_TOKEN = config_value("HF_TOKEN", "")
MAX_WORKER_JOBS_RAW = str(config_value("MAX_WORKER_JOBS", "auto"))
KEEPALIVE_LOG_SECONDS = int(config_value("KEEPALIVE_LOG_SECONDS", "60"))
KAGGLE_ACCELERATOR = config_value("KAGGLE_ACCELERATOR", "NvidiaTeslaT4")
VLLM_MODEL_ID = config_value("VLLM_MODEL_ID", MODEL_ID)
VLLM_SERVED_MODEL = config_value("VLLM_SERVED_MODEL", SERVED_MODEL)
VLLM_HOST = config_value("VLLM_HOST", "127.0.0.1")
VLLM_PORT = int(config_value("VLLM_PORT", "8001"))
VLLM_QUANTIZATION = config_value("VLLM_QUANTIZATION", "")
VLLM_TENSOR_PARALLEL_SIZE_RAW = str(config_value("VLLM_TENSOR_PARALLEL_SIZE", "auto"))
VLLM_MAX_MODEL_LEN = int(config_value("VLLM_MAX_MODEL_LEN", "4096"))
VLLM_GPU_MEMORY_UTILIZATION = str(config_value("VLLM_GPU_MEMORY_UTILIZATION", "0.88"))
VLLM_DTYPE = config_value("VLLM_DTYPE", "auto")
VLLM_TRUST_REMOTE_CODE = config_value("VLLM_TRUST_REMOTE_CODE", "false").lower() in {"1", "true", "yes", "y"}
VLLM_STARTUP_TIMEOUT_SECONDS = int(config_value("VLLM_STARTUP_TIMEOUT_SECONDS", "900"))
VLLM_EXTRA_ARGS = config_value("VLLM_EXTRA_ARGS", "")

if WORKER_BACKEND not in {"transformers", "vllm"}:
    raise RuntimeError(f"Unsupported WORKER_BACKEND={WORKER_BACKEND!r}")
if WORKER_BACKEND == "vllm":
    print({"event": "install_vllm"}, flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm"])

import torch
import websockets
if WORKER_BACKEND == "transformers":
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer

if WORKER_TOKEN and "token=" not in GATEWAY_WS_URL:
    separator = "&" if "?" in GATEWAY_WS_URL else "?"
    GATEWAY_WS_URL = f"{GATEWAY_WS_URL}{separator}token={WORKER_TOKEN}"

gateway_ws_parts = urlsplit(GATEWAY_WS_URL)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available; this worker requires a Kaggle GPU runtime")
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
GPU_COUNT = torch.cuda.device_count()
if MAX_WORKER_JOBS_RAW.lower() == "auto":
    MAX_WORKER_JOBS = GPU_COUNT if WORKER_BACKEND == "vllm" else 1
else:
    MAX_WORKER_JOBS = int(MAX_WORKER_JOBS_RAW)
MAX_WORKER_JOBS = max(1, MAX_WORKER_JOBS)
VLLM_TENSOR_PARALLEL_SIZE = GPU_COUNT if VLLM_TENSOR_PARALLEL_SIZE_RAW.lower() == "auto" else int(VLLM_TENSOR_PARALLEL_SIZE_RAW)
VLLM_TENSOR_PARALLEL_SIZE = max(1, VLLM_TENSOR_PARALLEL_SIZE)
unsupported = [name for name in gpu_names if "P100" in name]
if unsupported:
    raise RuntimeError(f"Unsupported Kaggle GPU for current PyTorch build: {unsupported}. Push with --accelerator NvidiaTeslaT4.")

print({"node_id": NODE_ID, "backend": WORKER_BACKEND, "served_model": SERVED_MODEL, "model_id": MODEL_ID, "vllm_model_id": VLLM_MODEL_ID, "vllm_quantization": VLLM_QUANTIZATION, "vllm_tensor_parallel_size": VLLM_TENSOR_PARALLEL_SIZE, "load_in_4bit": LOAD_IN_4BIT, "hf_token_set": bool(HF_TOKEN), "keepalive_log_seconds": KEEPALIVE_LOG_SECONDS, "requested_accelerator": KAGGLE_ACCELERATOR, "gateway_ws_scheme": gateway_ws_parts.scheme, "gateway_ws_host": gateway_ws_parts.netloc, "gpu_names": gpu_names}, flush=True)

In [ ]:
hub_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
tokenizer = None
model = None
vllm_process = None
vllm_base_url = f"http://{VLLM_HOST}:{VLLM_PORT}"


def vllm_command():
    command = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        VLLM_MODEL_ID,
        "--served-model-name",
        VLLM_SERVED_MODEL,
        "--host",
        VLLM_HOST,
        "--port",
        str(VLLM_PORT),
        "--tensor-parallel-size",
        str(VLLM_TENSOR_PARALLEL_SIZE),
        "--max-model-len",
        str(VLLM_MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        VLLM_GPU_MEMORY_UTILIZATION,
        "--generation-config",
        "vllm",
    ]
    if VLLM_QUANTIZATION:
        command.extend(["--quantization", VLLM_QUANTIZATION])
    if VLLM_DTYPE:
        command.extend(["--dtype", VLLM_DTYPE])
    if VLLM_TRUST_REMOTE_CODE:
        command.append("--trust-remote-code")
    if VLLM_EXTRA_ARGS:
        command.extend(shlex.split(VLLM_EXTRA_ARGS))
    return command


async def wait_for_vllm_ready():
    deadline = time.monotonic() + VLLM_STARTUP_TIMEOUT_SECONDS
    async with httpx.AsyncClient(timeout=10) as client:
        while time.monotonic() < deadline:
            if vllm_process and vllm_process.poll() is not None:
                raise RuntimeError(f"vLLM exited early with code {vllm_process.returncode}")
            try:
                response = await client.get(f"{vllm_base_url}/health")
                if response.status_code == 200:
                    return
            except Exception:
                pass
            await asyncio.sleep(5)
    raise TimeoutError(f"Timed out waiting for vLLM at {vllm_base_url}")


async def start_vllm_server():
    global vllm_process
    env = os.environ.copy()
    if HF_TOKEN:
        env["HF_TOKEN"] = HF_TOKEN
        env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    command = vllm_command()
    print({"event": "start_vllm", "command": " ".join(command), "base_url": vllm_base_url}, flush=True)
    vllm_process = subprocess.Popen(command, env=env)
    await wait_for_vllm_ready()
    print({"event": "vllm_ready", "base_url": vllm_base_url}, flush=True)


def stop_vllm_server():
    if vllm_process and vllm_process.poll() is None:
        vllm_process.terminate()


if WORKER_BACKEND == "transformers":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, **hub_kwargs)
    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
    }
    if LOAD_IN_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs["dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs, **hub_kwargs)
    model.eval()
    print("transformers model loaded")
else:
    await start_vllm_server()


In [ ]:
def build_prompt(messages):
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return "\n".join(f"{m['role']}: {m['content']}" for m in messages) + "\nassistant:"


async def send_json_locked(ws, send_lock, payload):
    async with send_lock:
        await ws.send(json.dumps(payload))


def make_generation_kwargs(inputs, job, streamer=None):
    temperature = float(job.get("temperature", 0.7))
    kwargs = dict(
        **inputs,
        max_new_tokens=int(job.get("max_tokens", 512)),
        do_sample=temperature > 0,
        pad_token_id=tokenizer.eos_token_id,
    )
    if temperature > 0:
        kwargs["temperature"] = temperature
        kwargs["top_p"] = float(job.get("top_p", 0.9))
    if streamer is not None:
        kwargs["streamer"] = streamer
    return kwargs


def generate_final_text(inputs, job):
    with torch.inference_mode():
        output_ids = model.generate(**make_generation_kwargs(inputs, job))
    prompt_length = inputs["input_ids"].shape[-1]
    new_tokens = output_ids[0][prompt_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


async def generate_for_job_transformers(ws, send_lock, job):
    job_id = job["job_id"]
    should_stream = bool(job.get("stream"))
    await send_json_locked(ws, send_lock, {"type": "job_started", "job_id": job_id})
    try:
        prompt = build_prompt(job["messages"])
        inputs = tokenizer(prompt, return_tensors="pt")
        first_device = next(model.parameters()).device
        inputs = {key: value.to(first_device) for key, value in inputs.items()}
        if not should_stream:
            content = await asyncio.to_thread(generate_final_text, inputs, job)
            await send_json_locked(ws, send_lock, {"type": "job_done", "job_id": job_id, "content": content})
            return

        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        kwargs = make_generation_kwargs(inputs, job, streamer=streamer)
        thread = Thread(target=model.generate, kwargs=kwargs)
        thread.start()
        parts = []
        while thread.is_alive():
            has_text, text = await asyncio.to_thread(next_stream_text, streamer)
            if not has_text:
                break
            parts.append(text)
            await send_json_locked(ws, send_lock, {"type": "token_delta", "job_id": job_id, "content": text})
            await asyncio.sleep(0)
        for text in streamer:
            parts.append(text)
            await send_json_locked(ws, send_lock, {"type": "token_delta", "job_id": job_id, "content": text})
        thread.join()
        await send_json_locked(ws, send_lock, {"type": "job_done", "job_id": job_id, "content": "".join(parts)})
    except Exception as exc:
        await send_json_locked(ws, send_lock, {
            "type": "job_error",
            "job_id": job_id,
            "error": str(exc),
            "traceback": traceback.format_exc(limit=4),
        })


def next_stream_text(streamer):
    try:
        return True, next(streamer)
    except StopIteration:
        return False, ""


def vllm_request_payload(job, should_stream):
    return {
        "model": VLLM_SERVED_MODEL,
        "messages": job["messages"],
        "temperature": float(job.get("temperature", 0.7)),
        "top_p": float(job.get("top_p", 0.9)),
        "max_tokens": int(job.get("max_tokens", 512)),
        "stream": should_stream,
    }


async def generate_for_job_vllm(ws, send_lock, job):
    job_id = job["job_id"]
    should_stream = bool(job.get("stream"))
    await send_json_locked(ws, send_lock, {"type": "job_started", "job_id": job_id})
    timeout_seconds = float(job.get("timeout", 600)) + 30
    timeout = httpx.Timeout(timeout_seconds, connect=30, read=timeout_seconds, write=30, pool=30)
    payload = vllm_request_payload(job, should_stream)
    url = f"{vllm_base_url}/v1/chat/completions"
    try:
        async with httpx.AsyncClient(timeout=timeout) as client:
            if not should_stream:
                response = await client.post(url, json=payload)
                response.raise_for_status()
                result = response.json()
                content = result.get("choices", [{}])[0].get("message", {}).get("content") or ""
                done = {"type": "job_done", "job_id": job_id, "content": content}
                if result.get("usage"):
                    done["usage"] = result["usage"]
                await send_json_locked(ws, send_lock, done)
                return

            parts = []
            usage = None
            async with client.stream("POST", url, json=payload) as response:
                if response.status_code >= 400:
                    body = (await response.aread()).decode("utf-8", errors="replace")
                    raise RuntimeError(f"vLLM HTTP {response.status_code}: {body[:500]}")
                async for line in response.aiter_lines():
                    if not line.startswith("data:"):
                        continue
                    data = line.removeprefix("data:").strip()
                    if not data:
                        continue
                    if data == "[DONE]":
                        break
                    chunk = json.loads(data)
                    if chunk.get("usage"):
                        usage = chunk["usage"]
                    choices = chunk.get("choices") or []
                    if not choices:
                        continue
                    text = (choices[0].get("delta") or {}).get("content") or ""
                    if text:
                        parts.append(text)
                        await send_json_locked(ws, send_lock, {"type": "token_delta", "job_id": job_id, "content": text})
                        await asyncio.sleep(0)
            done = {"type": "job_done", "job_id": job_id, "content": "".join(parts)}
            if usage:
                done["usage"] = usage
            await send_json_locked(ws, send_lock, done)
    except Exception as exc:
        await send_json_locked(ws, send_lock, {
            "type": "job_error",
            "job_id": job_id,
            "error": str(exc),
            "traceback": traceback.format_exc(limit=4),
        })


async def generate_for_job(ws, send_lock, job):
    if WORKER_BACKEND == "vllm":
        await generate_for_job_vllm(ws, send_lock, job)
    else:
        await generate_for_job_transformers(ws, send_lock, job)


async def run_job_with_capacity(ws, send_lock, job_semaphore, job):
    async with job_semaphore:
        await generate_for_job(ws, send_lock, job)


async def heartbeat(ws, send_lock):
    while True:
        await asyncio.sleep(15)
        await send_json_locked(ws, send_lock, {"type": "heartbeat", "node_id": NODE_ID})


async def keepalive_log():
    while True:
        await asyncio.sleep(KEEPALIVE_LOG_SECONDS)
        print({"event": "worker_keepalive", "node_id": NODE_ID, "time": int(time.time())}, flush=True)


async def run_worker():
    while True:
        try:
            async with websockets.connect(GATEWAY_WS_URL, max_size=None, ping_interval=20) as ws:
                send_lock = asyncio.Lock()
                job_semaphore = asyncio.Semaphore(MAX_WORKER_JOBS)
                active_tasks = set()
                await send_json_locked(ws, send_lock, {
                    "type": "register",
                    "node_id": NODE_ID,
                    "owner": OWNER,
                    "model": SERVED_MODEL,
                    "accelerator": "NvidiaTeslaT4x2",
                    "capacity": MAX_WORKER_JOBS,
                })
                print("registered", await ws.recv(), flush=True)
                heartbeat_task = asyncio.create_task(heartbeat(ws, send_lock))
                try:
                    async for raw in ws:
                        message = json.loads(raw)
                        if message.get("type") == "job":
                            task = asyncio.create_task(run_job_with_capacity(ws, send_lock, job_semaphore, message))
                            active_tasks.add(task)
                            task.add_done_callback(active_tasks.discard)
                        elif message.get("type") == "terminate":
                            reason = message.get("reason", "terminated by root")
                            print({"event": "worker_terminate", "node_id": NODE_ID, "reason": reason}, flush=True)
                            await send_json_locked(ws, send_lock, {"type": "terminate_ack", "node_id": NODE_ID, "reason": reason})
                            for task in active_tasks:
                                task.cancel()
                            await ws.close()
                            return
                finally:
                    heartbeat_task.cancel()
                    for task in active_tasks:
                        task.cancel()
                    if active_tasks:
                        await asyncio.gather(*active_tasks, return_exceptions=True)
        except Exception as exc:
            print("worker reconnect after error", repr(exc), flush=True)
            await asyncio.sleep(10)


worker_task = asyncio.create_task(run_worker())
keepalive_task = asyncio.create_task(keepalive_log())
try:
    await worker_task
finally:
    keepalive_task.cancel()
    stop_vllm_server()
    print({"event": "worker_stopped", "node_id": NODE_ID}, flush=True)